# DS4DS Exercise Sheet 11

**General Instructions:**

- Review the weekly course material (lectures, readings, slides, etc.) before starting the exercises.
- Complete all assigned exercises independently before the Q&A session.
- Please use Julia Version 1.12.x to ensure compatibility.
- Please only write between the `#--- YOUR CODE STARTS HERE ---#` and `#--- YOUR CODE ENDS HERE ---#` comments.

Run the following cell 'ONLY' if you don't use the .toml files provided with the project.

In [ ]:
# Notebook-local environment bootstrap for Julia 1.12.1 (one-stop)
# This cell activates the notebook folder as a project, installs the required packages
# and attempts to build & precompile them. Run this cell first (it may take a few minutes).
import Pkg
proj_dir = dirname(Base.current_project())
println("Ensuring project at: ", proj_dir)
try
    Pkg.activate(proj_dir)
    # Minimal package set required by the notebook (tweak if you know additional needs)
    pkgs = [
        "Printf", "DelimitedFiles", "Plots", "GZip", "StatsBase", "LaTeXStrings",
        "Serialization", "OrdinaryDiffEq", "Lux", "Random", "ComponentArrays",
        "SciMLSensitivity", "Optimization", "Zygote", "Flux"
    ]
    for p in pkgs
        try
            println("Adding package: ", p)
            Pkg.add(p)
        catch e
            @warn "Pkg.add failed for $p; continuing" exception=(e, catch_backtrace())
        end
    end
    try
        Pkg.instantiate()
        Pkg.precompile()
    catch e
        @warn "Instantiate / precompile failed (continuing)." exception=(e, catch_backtrace())
    end
catch e
    @warn "Failed to activate notebook project." exception=(e, catch_backtrace())
end

In [ ]:
using Printf
using DelimitedFiles
using Plots
using GZip
using StatsBase
using LaTeXStrings
using Serialization
using OrdinaryDiffEq
using Lux
using Random
using ComponentArrays
# SciMLSensitivity provides reverse-mode adjoint support required by Optimization + Zygote
using SciMLSensitivity
using Optimization
# We no longer depend on OptimizationFlux; training uses a Flux-based loop instead
using Zygote
using Flux
# NOTE: removed other unused / optional packages to reduce dependencies and improve compatibility with Julia 1.12.1
# Removed earlier: DataInterpolations, OptimizationOptimJL, LineSearches

#### **a)** Split your dataset into a training and evaluation dataset. The training dataset will be used to tune the parameters of your model and the evaluation data will be used to evaluate the generalization performance of your model. Use $u(t >= 37.5s)$ and $y(t >= 37.5s)$ for the training and $u(t < 37.5s)$ and $y(t < 37.5s)$ for the evaluation datasets. Additionally, provide the corresponding timesteps for the training and eval datasets, i.e., provide ```tsteps_training``` which contains all timesteps from the dataset with $t >= 37.5s$ and ```tsteps_eval``` which contains all timesteps with $t < 37.5s$.

Implement the function below to split the dataset.

In [ ]:
function loaddata_heating(; normalize=true)
    fh = GZip.open("heating_system.dat.gz")
    data = readdlm(fh)

    data_freq = 4
    
    data = data[1:data_freq:end, :]
    
    t = data[:, 1] / 16  # time in seconds
    u = data[:, 2]  # input: voltage driving the lamp in volt
    y = data[:, 3]  # output: temperature measurement in °C

    
    dt = 2 * data_freq / 16  # time between samples in seconds (thermic process -> high inertia)
    
    # convert to Float32 for performance
    t = Float32.(t)
    u = Float32.(u) 
    y = Float32.(y)
    
    if normalize  # optional standardization of data
        dty = fit(ZScoreTransform, y)
        dtu = fit(ZScoreTransform, u)
    
        y = StatsBase.transform(dty, y)
        u = StatsBase.transform(dtu, u);
    end
    
    tspan = (t[1], t[end])
    
    return u, y, dt, tspan, t
end

In [ ]:
u, y, dt, tspan, tsteps = loaddata_heating();

function split_dataset(u, y, tsteps)
    """
    Creates a training and evaluation split
    
    Args:
        u: System input data
        y: System output data
        tsteps: time steps of the data trajectories

    Returns:
        u_training: u(t >= 37.5s)
        y_training: y(t >= 37.5s)
        u_eval: u(t < 37.5s)
        y_eval: y(t < 37.5s)
        tsteps_training: time steps for the training data
        tsteps_eval: time steps for the eval data
    """
    ### BEGIN SOLUTION

    belonging_to_training = tsteps .>= 37.5
    belonging_to_eval = tsteps .< 37.5
    
    u_training = u[belonging_to_training]
    y_training = y[belonging_to_training]
    tsteps_training = tsteps[belonging_to_training]
    
    u_eval = u[belonging_to_eval]
    y_eval = y[belonging_to_eval]
    tsteps_eval = tsteps[belonging_to_eval]
    
    ### END SOLUTION

    return u_training, y_training, u_eval, y_eval, tsteps_training, tsteps_eval
end
    
u_training, y_training, u_eval, y_eval, tsteps_training, tsteps_eval = split_dataset(u, y, tsteps);

In [ ]:
u, y, dt, tspan, tsteps = loaddata_heating(normalize = false)
p = plot(tsteps, u, label=L"u", title="Simple heating system", xlabel=L"$t$ in seconds", ylabel=L"$u$ in V / $y$ in °C",
     legend=:topleft, lw=2, palette = :Set2_5)
plot!(p, tsteps, y, label=L"y", lw =2)

#### **b)** Implement a function for the creation of a fully-connected feed-forward neural network in dependence of the layer sizes and the activation functions. Use ```Lux.jl``` to implement it in the function template given below.

In [ ]:
function create_ann(layer_sizes, hidden_layer_activation_function, output_activation)
    """
    Creates a neural network with the given parameters

    Args:
        layer_sizes: List of layer sizes (e.g. [5, 32, 32, 2])
        hidden_layer_activation_function: Activation function for all hidden layers (nomenclature of Lux.jl)
        output_activation: Activation function of the output layer (nomenclature of Lux.jl)

    Returns:
        ann: Resulting fully-connected neural network
    """
    ### BEGIN SOLUTION

    chain_elements = [Lux.Dense(m, n, hidden_layer_activation_function) for (m, n) in zip(layer_sizes[1:end-2], layer_sizes[2:end-1])]
    chain_elements = vcat(chain_elements, Lux.Dense(layer_sizes[end-1], layer_sizes[end], output_activation))
    ann = Lux.Chain(chain_elements)

    ### END SOLUTION

    return ann
end

# the sizes you choose here have no influence on the grading of this subtask
layer_sizes = nothing
hidden_layer_activation_function = nothing
output_activation = nothing

### BEGIN SOLUTION
layer_sizes = [2, 40, 40, 1]
hidden_layer_activation_function = tanh
output_activation = x -> x
### END SOLUTION

In [ ]:
ann = create_ann(layer_sizes, hidden_layer_activation_function, output_activation)
@assert isa(ann, Lux.Chain)

### BEGIN TESTS
test_ann = create_ann([2, 40, 40, 1], tanh, x -> x)

@assert length(test_ann) == 3

@assert test_ann[1].in_dims == 2
@assert test_ann[1].out_dims == 40

@assert test_ann[2].in_dims == 40
@assert test_ann[2].out_dims == 40

@assert test_ann[3].in_dims == 40
@assert test_ann[3].out_dims == 1

@assert isa(test_ann.layers.layer_1.activation, Function)
@assert isa(test_ann.layers.layer_2.activation, Function)
@assert isa(test_ann.layers.layer_3.activation, Function)

### END TESTS

#### **c)** In this subtask, we want to define the NODE of the form

$$
\begin{align}
    \frac{\mathrm{d}}{\mathrm{d}t} \hat{x}(t) &= \mathcal{M}_{\mathbf{w}} (\hat{x}(t), u(t)).
\end{align}
$$

To this end, we need to define the ODE in the function ```dxdt(x, w, t)``` and initialize the ```training_problem```.
Use ```create_ann``` from subtask **b)** to create a model for the ODE. If you were unable to properly implement it in **b)**, you are also allowed to create a model manually. However, you are restricted to fully-connected feed-forward neural networks. Use the variable ```w``` for the model parameters $\mathbf{w}$.

The layer sizes, activation functions and initialization are set here to help you with the following subtask.
If you want to design a different network, you are free to do so. Note, however that you might face a more complex optimization in that case.

In [ ]:
# theoretically you would need to interpolate separately for training and evaluation, but we will look past this here
# as the time is already dividing between the two
# Use a small, dependency-free linear interpolator to avoid compatibility issues
function make_linear_interp(x, y)
    return t -> begin
        if t <= x[1]
            return y[1]
        elseif t >= x[end]
            return y[end]
        else
            i = searchsortedfirst(x, t)
            x0 = x[i-1]; x1 = x[i]
            y0 = y[i-1]; y1 = y[i]
            return y0 + (y1 - y0)*(t - x0)/(x1 - x0)
        end
    end
end
s_interp = make_linear_interp(tsteps, u)  # capture local scalar interpolator for SISO

# For this task, we have:
# - 1 state variable (the temperature)
# - 1 input variable (voltage)
# So total input size = 2
input_size = 2  # state dimension (1) + input dimension (1)
hidden_size = 8
output_size = 1
ann = create_ann([input_size, hidden_size, output_size], tanh, x -> x)

# initialize parameter and state variables
seed = 2  # This seed gives good initial state for optimization
w, st = Lux.setup(MersenneTwister(seed), ann);
w = ComponentArray(w);

### BEGIN SOLUTION
# define dxdt using a safe input construction to avoid shape mismatches
function dxdt(x, w, t)
    xin = vec(x)
    T = eltype(xin)
    u_val = T(s_interp(t))  # ensure input shares the element type (supports dual numbers)
    inp = vcat(xin, u_val)  # Concatenate state and input
    y_hat, _ = Lux.apply(ann, inp, w, st)  # Forward pass without mutating state
    return y_hat
end

x0 = [Float32(y_training[1])]  # Initial state as Vector{Float32}
training_prob = ODEProblem(dxdt, x0, (Float32(tsteps_training[1]), Float32(tsteps_training[end])), w)
### END SOLUTION

#### **d)** Next, we want to implement the forward pass of the NODE.

Implement the function below that takes the parameters of the model $\mathbf{w}$, an ODE problem, the initial state $x_0$ and the timesteps and returns the state trajectory $\hat{\mathbf{x}}_n$.
Use the Tsit5 ODE solver, but non-adaptive.

In [ ]:
function predict_NODE(w, problem, x0, tsteps, dt)
    """
    NODE forward pass

    Args:
        w: Parameters of the model as a ComponentArray
        problem: ODE problem
        x0: Initial state of the trajectory
        tsteps: Time steps of the trajectory
        dt: The time between two samples in the trajectory

    Returns:
        Resulting state trajectory
    
    """
    ### BEGIN SOLUTION
    return Array(solve(problem, Tsit5(); u0=x0, p=w, saveat=tsteps, dt, adaptive=false))
    ### END SOLUTION
end;

In [ ]:
@assert isa(predict_NODE, Function)

### BEGIN TESTS

dummy_ode(x, w, t) = w * x
test_x0 = 2
test_w = 0.5

test_dt = 0.1
test_tsteps = LinRange(0, 1, 10)

trajectory = predict_NODE(
    test_w,
    ODEProblem(dummy_ode, test_x0, (test_tsteps[1], test_tsteps[end]), test_w),
    test_x0,
    test_tsteps,
    test_dt
)

println("Given trajectory: ", trajectory)

expected_trajectory = [2.0, 2.114255489520473, 2.2350381374837274, 2.362720825731292, 2.497697738003365, 2.640385576868242, 2.7912248501721817, 2.950681230981243, 3.119246995213576, 3.297442541400258]
@assert isapprox(expected_trajectory, trajectory, rtol=0, atol=1e-6)

### END TESTS

#### **e)** Implement the MSE loss function to compare model predictions and true trajectory. Optimize the parameters $\mathbf{w}$ of your model given in subtask c) using the given optimizer. Design an adequate optimization problem for the given template.


In [ ]:
# Task 1 training utilities (train fresh weights on every run)
using Flux.Losses: mse
maxiters_task1 = 2_000

### BEGIN SOLUTION
function loss_NODE_task1_eval(w_params)
    pred = predict_NODE(w_params, training_prob, x0, tsteps_training, dt)
    return mse(vec(pred), y_training)
end
### END SOLUTION

function train_task1!(; maxiters::Int = maxiters_task1, lr::Float64 = 1e-3, patience::Int = 15, tol::Float64 = 1e-6)
    current_w = deepcopy(w)
    best_w = deepcopy(current_w)
    best_loss = loss_NODE_task1_eval(current_w)
    no_improve = 0
    lr_val = eltype(current_w)(lr)
    for iter in 1:maxiters
        ### BEGIN SOLUTION
        # 1. calculate the loss gradient w.r.t. to parameters w
        grad_w = Zygote.gradient(loss_NODE_task1_eval, current_w)[1]
        # 2. update the parameters w
        current_w .= current_w .- lr_val .* grad_w
        ### END SOLUTION
        if iter % 50 == 0
            current_loss = loss_NODE_task1_eval(current_w)
            @printf("Iteration: %5d, Training loss: %.6e\n", iter, current_loss)
            if current_loss + tol < best_loss
                best_loss = current_loss
                best_w = deepcopy(current_w)
                no_improve = 0
            else
                no_improve += 1
            end
            if no_improve >= patience
                break
            end
        end
    end
    w .= best_w
    return ComponentArray(best_w)
end

After you have trained your model, continue with the evaluation below. The notebook now trains fresh weights on every run and keeps them in memory only.

In [ ]:
# Train Task 1 model and keep weights in memory
student_w = train_task1!()
@assert student_w !== nothing

#### **f)** Evaluate the performance of your model on the evaluation dataset. To do so, define the evaluation problem.

Produce a model that reaches an MSE between the prediction and the eval trajectory lower than the threshold given below.
The previously given architecture and optimization parameters should help you reach this prediction accuracy.

In [ ]:
x0_eval = nothing
evaluation_problem = nothing

### BEGIN SOLUTION
x0_eval = [Float32(y_eval[1])]
evaluation_problem = ODEProblem(dxdt, x0_eval, (Float32(tsteps_eval[1]), Float32(tsteps_eval[end])), student_w)
### END SOLUTION

In [ ]:
@assert isa(x0_eval, AbstractVector)
@assert isa(evaluation_problem, ODEProblem)

prediction_eval = vec(predict_NODE(student_w, evaluation_problem, x0_eval, tsteps_eval, dt))

@assert length(prediction_eval) == length(tsteps_eval)

prediction_error_eval = Flux.mse(prediction_eval, y_eval)
println("Task 1 prediction error on eval set: ", prediction_error_eval)

@assert prediction_error_eval <= 0.3

In [ ]:
prediction_training = vec(predict_NODE(student_w, training_prob, x0, tsteps_training, dt))

p = plot(tsteps_training, y_training, label=L"y", lw =2, title="Training set performance")
plot!(p, tsteps_training, u_training, label=L"u", lw =2)
plot!(p, tsteps_training, prediction_training, label=L"\hat{y}", lw =2)

In [ ]:
eval_plot = scatter(tsteps_eval, y_eval, label=L"$y(t)$", title="Eval performance", xlabel=L"$t$ in seconds", ylabel=L"$y$ in °C", legend=:bottomleft, lw=2)
plot!(eval_plot, tsteps_eval, prediction_eval, label=L"$\hat{y}(t)$", lw =2)

## Task 2: NODE model for Lake Erie (MIMO)

In this task, we once again model an underlying process using NODEs. However, the system at hand now takes multiple inputs $\mathbf{u}(t)$ and yields multiple outputs $\mathbf{y}(t)$. This means we need to model a multiple-input-multiple-output (MIMO) system.

This results in the dynamics equations 

$$
\begin{align}
    \frac{\mathrm{d}}{\mathrm{d}t} \hat{\mathbf{x}}(t) &= \mathcal{M}_{\mathbf{w}} (\hat{\mathbf{x}}(t), \mathbf{u}(t)) \\
    \hat{\mathbf{y}}(t) & = \hat{\mathbf{x}} (t).
\end{align}
$$

The data ([source](https://ftp.esat.kuleuven.be/pub/SISTA/data/environmental/) and [description](https://ftp.esat.kuleuven.be/pub/SISTA/data/environmental/erie.txt) of the dataset) is the result of a simulation of the western basin of Lake Erie. The inputs $\mathbf{u}(t)$ are the water temperature, water conductivity, water alkalinity, the NO3 content and the total hardness of the water. The outputs $\mathbf{y}(t)$ are the amount of dissolved oxigen and the algae content. We are interested in the dynamic relationship between inputs and outputs described by the model above.

#### a) Below you are shown the raw output data:

![lake_erie-not_normalized](./LakeErie-NotNormalized.svg)

You can see that the two outputs are scaled very differently. As a result, we need to normalize them separately.

Implement the function below to normalize each output separately. Use `StatsBase.fit(ZScoreTransform, ...)` and `StatsBase.transform(...)` for the normalization.

In [ ]:
function normalize_columnwise(data)

    ### BEGIN SOLUTION

    normalized_data = zeros(size(data))
    for i in 1:size(data)[2]
        column = Float32.(data[:, i])
        
        dt_column = StatsBase.fit(ZScoreTransform, column)
        normalized_data[:, i] = StatsBase.transform(dt_column, column)
    end
    
    ### END SOLUTION
    
    return normalized_data
end

function loaddata_erie(; normalize= true)
    fh = GZip.open("erie.dat.gz")
    data = readdlm(fh)
    
    t = data[:, 1]  # time in months
    u = data[:, 2:6]
    y = data[:, 22:23]
    
    dt = 1  # time between samples in months
    
    # convert to Float32 for performance
    t = Float32.(t)
    u = Float32.(u) 
    y = Float32.(y)
    
    if normalize  # optional standardization of data
        y = normalize_columnwise(y)
        u = normalize_columnwise(u)
    end
    
    tspan = (t[1], t[end])
    
    return u, y, dt, tspan, t
end

Code for plot below when the data is normalized properly:
```
p = plot(tsteps, y, label=L"y", title="Lake Erie", xlabel=L"$t$ in months", ylabel=L"y", legend=:topleft, background_color="#000000",lw=2, palette = :Set2_5)
```

![lake_erie](./LakeErie.svg)

In [ ]:
isa(normalize_columnwise, Function)

### BEGIN TESTS

test_data = reduce(hcat, [LinRange(-1, 1, 10), LinRange(-5, 0, 10), LinRange(12, 4596, 10)])

normalized_data = normalize_columnwise(test_data)

expected_column = [-1.4863, -1.15601, -0.825723, -0.495434, -0.165145, 0.165145, 0.495434, 0.825723, 1.15601, 1.4863]

for idx in 1:size(normalized_data, 2)
    norm_column = normalized_data[:, idx]
    println("Given column: ", norm_column)
    @assert isapprox(norm_column, expected_column, rtol=0, atol=1e-4) 
end

### END TESTS

#### Preparations:

In [ ]:
u, y, dt, tspan, tsteps = loaddata_erie(normalize=true)
print(size(u))

# interpolate inputs for all channels separately using the small linear interpolator
interp_input = [make_linear_interp(tsteps, u[:, i]) for i in 1:size(u, 2)]
function interpolate_input(t)
    return [f(t) for f in interp_input]
end

In [ ]:
belonging_to_training = tsteps .<= 40
belonging_to_eval = tsteps .> 40

u_training = u[belonging_to_training, :]
y_training = y[belonging_to_training, :]
tsteps_training = tsteps[belonging_to_training, :]

u_eval = u[belonging_to_eval, :]
y_eval = y[belonging_to_eval, :]
tsteps_eval = tsteps[belonging_to_eval, :];

tsteps_training = vec(tsteps_training)
tsteps_eval = vec(tsteps_eval);

#### **b)** Build and train a NODE model using the training dataset. Design a feed-forward fully-connected neural network as the model.

In [ ]:
### BEGIN SOLUTION
seed = 0

# State dimension (2) + input dimension (5) = total input size (7)
state_dim = 2
input_dim = 5
layer_sizes = [state_dim + input_dim, 16, state_dim]
hidden_layer_activation_function = tanh
output_activation = x -> x
ann = create_ann(layer_sizes, hidden_layer_activation_function, output_activation)

x0_task2 = y_training[1, :]
w_erie, st = Lux.setup(MersenneTwister(seed), ann);
w_erie = ComponentArray(w_erie);

# capture the MIMO interpolator closure so later code doesn't overwrite globals
m_interp = interpolate_input

# define dxdt: build a flat input vector (state concatenated with interpolated inputs)
function dxdt(x, w, t)
    xin = vec(x)
    T = eltype(xin)
    u_vec = T.(m_interp(t))
    inp = vcat(xin, u_vec)
    y_hat, _ = Lux.apply(ann, inp, w, st)
    return y_hat
end

training_prob_task2 = ODEProblem(dxdt, x0_task2, (Float32(tsteps_training[1]), Float32(tsteps_training[end])), w_erie)

using Flux.Losses: mse

function loss_NODE_task2_eval(w_params)
    pred = transpose(predict_NODE(w_params, training_prob_task2, x0_task2, tsteps_training, dt))
    return mse(pred, y_training)
end

function evaluate_result(w_params, y_eval_data, tsteps_eval_data)
    x0_eval_local = y_eval_data[1, :]
    evaluation_prob_local = ODEProblem(dxdt, x0_eval_local, (Float32(tsteps_eval_data[1]), Float32(tsteps_eval_data[end])), w_params)
    prediction_eval_local = transpose(predict_NODE(w_params, evaluation_prob_local, x0_eval_local, tsteps_eval_data, dt))
    prediction_error_local = mse(prediction_eval_local, y_eval_data)
    println("Task 2 eval MSE: ", prediction_error_local)
    return prediction_eval_local, prediction_error_local
end

function train_task2!(; maxiters::Int = 200, lr::Float64 = 5e-4, patience::Int = 20, tol::Float64 = 1e-6)
    current_w = deepcopy(w_erie)
    best_w = deepcopy(current_w)
    best_loss = loss_NODE_task2_eval(current_w)
    no_improve = 0
    lr_val = eltype(current_w)(lr)
    for iter in 1:maxiters
        grad_w = Zygote.gradient(loss_NODE_task2_eval, current_w)[1]
        current_w .= current_w .- lr_val .* grad_w
        if iter % 50 == 0
            current_loss = loss_NODE_task2_eval(current_w)
            @printf("Iteration: %5d, Training loss: %.6e\n", iter, current_loss)
            if current_loss + tol < best_loss
                best_loss = current_loss
                best_w = deepcopy(current_w)
                no_improve = 0
            else
                no_improve += 1
            end
            if no_improve >= patience
                break
            end
        end
    end
    w_erie .= best_w
    return ComponentArray(best_w)
end
### END SOLUTION

After you have trained your model, proceed with the evaluation cells. The training routine runs each time the notebook is executed and does not write model parameters to disk.

#### **c)** Evaluate the performance of your model on the evaluation dataset. To do so, define the evaluation problem.

Produce a model that reaches an MSE between the prediction and the eval trajectory lower than the threshold given below. Design a model architecture and optimization routine in the previous subtasks that reaches this level of accuracy.

In [ ]:
# Train Task 2 model and keep weights in memory
student_w = train_task2!()
@assert student_w !== nothing

In [ ]:
x0_eval_task2 = nothing
evaluation_prob_task2 = nothing

### BEGIN SOLUTION
x0_eval_task2 = y_eval[1, :]
evaluation_prob_task2 = ODEProblem(dxdt, x0_eval_task2, (Float32(tsteps_eval[1]), Float32(tsteps_eval[end])), student_w)
### END SOLUTION

In [ ]:
#public test
@assert isa(x0_eval_task2, AbstractArray)
@assert isa(evaluation_prob_task2, ODEProblem)

prediction_eval_task2 = transpose(predict_NODE(student_w, evaluation_prob_task2, x0_eval_task2, tsteps_eval, dt))

@assert size(prediction_eval_task2, 1) == size(y_eval, 1)

prediction_error_eval_task2 = Flux.mse(prediction_eval_task2, y_eval)
println("Task 2 prediction error on eval set: ", prediction_error_eval_task2)



In [ ]:
prediction_training_task2 = transpose(predict_NODE(student_w, training_prob_task2, x0_task2, tsteps_training, dt))

p = plot(tsteps_training, y_training, label=L"y", title="Lake Erie", xlabel=L"$t$ in months", ylabel=L"y", legend=:topleft,lw=2, palette = :Set2_5)
# plot!(p, tsteps_training, u_training, label=L"u", lw =2)
plot!(p, tsteps_training, prediction_training_task2, label=L"\hat{y}", lw =2)

In [ ]:
prediction_eval_task2, prediction_error_eval_task2 = evaluate_result(student_w, y_eval, tsteps_eval)
eval_plot = Plots.plot(tsteps_eval, y_eval, label=L"$y(t)$", title="Lake Erie eval", xlabel=L"$t$ in months", ylabel=L"y", legend=:bottomleft,lw=2, palette = :Set2_5)
Plots.plot!(eval_plot, tsteps_eval, prediction_eval_task2, label=L"$\hat{y}(t)$", lw =2)
display(eval_plot)